# 🎈 PicoCAST Regional Multi-Radar Discovery Pipeline

This notebook runs the complete multi-radar regional discovery pipeline for balloon-like candidates. The pipeline includes S3 archives querying, visibility modeling, DBSCAN clustering, branching tracklet linking, cross-radar spatial-temporal association, and automated telemetry comparisons.

All time-consuming steps are equipped with **progress bars** for real-time tracking.

### Setup & Imports
We add the repository root to `sys.path` so we can import the pipeline modules directly.

In [1]:
import sys
from pathlib import Path

# Locate repo root relative to notebooks directory
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_PATH = "../cases/k7uaz_20260322/config.yaml"
print(f"Using config path: {CONFIG_PATH}")

Using config path: ../cases/k7uaz_20260322/config.yaml


## Phase 0 & 1: NEXRAD Level II Ingest & Geometry Dry-Run
Evaluates geometry first and outputs `regional_radar_geometry.csv`, then downloads Level II files from S3 in parallel.

In [2]:
from scripts.download_regional_nexrad import main as download_main

# 1. Dry run geometry check first
print("--- Running Phase 0 Geometry Dry-Run ---")
sys.argv = ["download_regional_nexrad.py", CONFIG_PATH, "--primary-sites", "--dry-run"]
download_main()

# 2. Real download/ingest with progress bar
print("\n--- Running Phase 1 Parallel Ingest ---")
sys.argv = ["download_regional_nexrad.py", CONFIG_PATH, "--primary-sites"]
download_main()

--- Running Phase 0 Geometry Dry-Run ---
Targeting radar sites: KEMX, KIWA, KFSX, KYUX, KEPZ
Evaluating radar geometries and listings...
Wrote geometry report: ../cases/k7uaz_20260322/nexrad/regional_radar_geometry.csv
radar_site  radar_lat  radar_lon  radar_alt_m  min_range_km  max_range_km  n_visible_scans  n_total_scans  visibility_fraction geometry_status                                       skip_reason
      KEMX   31.89365 -110.63025       1621.2         48.06         87.03               27             36               0.7500         include                                                  
      KIWA   33.28923 -111.66991        434.6        131.92        209.26               26             35               0.7429         include                                                  
      KFSX   34.57433 -111.19844       2290.3        251.83        288.89                0             29               0.0000            skip Radar is too far (min distance 251.8 km > 250 km)
      KYU

## Phase 2: Regional Scan Matching & 4/3 Earth Visibility Model
Calculates slant range and beam-center altitude sweeps ($0.5^\circ$ to $19.5^\circ$), matching scans to the expected flight corridor.

In [3]:
from scripts.match_regional_scans_to_telemetry import main as match_main

sys.argv = ["match_regional_scans_to_telemetry.py", CONFIG_PATH, "--primary-sites"]
match_main()

Matching telemetry for active sites: KEMX, KIWA
Reloading existing match index for KEMX: ../cases/k7uaz_20260322/nexrad/KEMX/index/scan_track_matches.csv
Reloading existing match index for KIWA: ../cases/k7uaz_20260322/nexrad/KIWA/index/scan_track_matches.csv
Wrote merged regional scan matches: ../cases/k7uaz_20260322/nexrad/regional_scan_matches.csv


## Phase 3: DBSCAN Candidate Cluster Discovery
Extracts compact balloon-like candidates within the corridor bounds ($±1500\text{ m}$ altitude, $\le 40\text{ km}$ horizontal offset) and saves them to Parquet format.

In [4]:
from scripts.discover_regional_balloon_like_clusters import main as discover_main

# Using --overwrite if you want to force re-clustering
sys.argv = ["discover_regional_balloon_like_clusters.py", CONFIG_PATH, "--primary-sites"]
discover_main()

Discovering clusters for active sites: KEMX, KIWA
Reloading existing candidate clusters for KIWA
Reloading existing candidate clusters for KEMX
Wrote merged regional clusters: ../cases/k7uaz_20260322/outputs/discovery/regional_discovered_clusters.parquet


## Phase 4: Branching Tracklet Linking
Runs branching forward search to link candidate points over time, allowing up to 2 missing scans and pruning duplicate overlapping paths.

In [5]:
from scripts.link_regional_tracklets import main as link_main

sys.argv = ["link_regional_tracklets.py", CONFIG_PATH, "--primary-sites"]
link_main()

Linking tracklets for active sites: KEMX, KIWA
Reloading existing tracklets for KEMX
Reloading existing tracklets for KIWA


## Phase 5: Cross-Radar Candidate Association
Interpolates coordinate trajectories from different radars to common overlap times and labels spatial associations (strong, moderate, weak).

In [6]:
from scripts.associate_cross_radar_tracklets import main as associate_main

sys.argv = ["associate_cross_radar_tracklets.py", CONFIG_PATH, "--primary-sites"]
associate_main()

Reloading existing associations: ../cases/k7uaz_20260322/outputs/discovery/cross_radar_tracklet_associations.csv


## Phase 6: Telemetry Comparisons
Rates how closely candidate tracklets match expected altitude profiles and ground track segments.

In [7]:
from scripts.compare_regional_tracklets_to_telemetry import main as compare_main

sys.argv = ["compare_regional_tracklets_to_telemetry.py", CONFIG_PATH, "--primary-sites"]
compare_main()

Reloading existing telemetry comparison: ../cases/k7uaz_20260322/outputs/discovery/regional_tracklet_telemetry_comparison.csv


## Phase 7: Interactive GIS & Profile Dashboard
Generates the dual-panel visualizer showing all tracklets, cross-radar lines, sweeps, and expected path altitude overlay.

In [8]:
from scripts.make_regional_discovery_dashboard import main as dashboard_main

sys.argv = ["make_regional_discovery_dashboard.py", CONFIG_PATH, "--primary-sites"]
dashboard_main()

Reloading existing dashboard: ../cases/k7uaz_20260322/outputs/discovery/regional_discovery_dashboard.html


## Phase 8 & 10: Scientific Report & Package Delivery
Compiles the final report markdown file and packages the event packet deliverables under `outputs/discovery/event_packet/` and `docs/discovery/`.

In [9]:
from scripts.write_regional_discovery_report import main as report_main

sys.argv = ["write_regional_discovery_report.py", CONFIG_PATH, "--primary-sites", "--overwrite"]
report_main()

Wrote regional discovery report: ../cases/k7uaz_20260322/outputs/discovery/regional_discovery_report.md
Packaging Phase 10 Event Packet at: ../cases/k7uaz_20260322/outputs/discovery/event_packet
Event packet packaging complete.
Copied event packet files to docs/discovery/
